# ConnectTel Customer Churn — Notebook 1: Data Cleaning & Train-Test Split

## Objective

This notebook performs **structural data cleaning** (fixing things that are true regardless of any statistical fitting — duplicates, inconsistent text labels, known sentinel/error codes, redundant columns, non-predictive IDs) and then performs the **train-test split immediately after**, *before* any preprocessing step that involves fitting (imputation, scaling, encoding, capping).

### Why split before preprocessing?

In the original pipeline, the train-test split happened inside `model_building.ipynb`, **after** the `ColumnTransformer` (imputer + scaler + one-hot encoder) had already been `fit_transform`-ed on the *entire* dataset in `feature_engineering.ipynb`. That means the median used for imputation, the mean/std used for scaling, and the categories seen by the encoder were all computed using information from what later became the test set. This is a classic **data leakage** issue — it makes test performance look better than it will be in production, because the test set indirectly influenced the transformations applied to it.

**Fix:** Split first. Every statistic used later (medians, means, standard deviations, IQR bounds, category frequencies) will be learned **only from the training set** and then applied to the test set, exactly like a model would treat truly unseen customers.

### What happens in this notebook
1. Load raw data and inspect it
2. Remove exact duplicate rows (safe — no statistical fitting involved)
3. Standardize inconsistent category labels (e.g. `postpaid` / `POSTPAND` → `Postpaid`)
4. Replace an obvious sentinel/error code in `monthly_data_usage_gb`
5. Drop non-predictive identifier columns and one redundant duplicate column
6. Perform the stratified train-test split
7. Save `train_raw.csv` and `test_raw.csv` for downstream notebooks


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)

RAW_PATH = Path("../data/raw/ConnectTel_Churn_Intelligence.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and inspect raw data

In [2]:
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
df.head()


Shape: (50500, 34)


,customer_id,customer_age,gender,city,customer_tenure_months,customer_segment,plan_type,monthly_bill_amount,contract_type,number_of_services,monthly_data_usage_gb,monthly_voice_minutes,monthly_sms_count,international_usage_flag,network_drop_rate,average_download_speed,service_outages_last_6m,support_tickets_last_12m,avg_resolution_time_hours,complaints_last_12m,mobile_app_logins_last_30d,reward_points_balance,campaign_response_rate,late_payments_last_12m,autopay_enabled,billing_disputes_last_12m,customer_lifetime_value,monthly_revenue,retention_offer_count,legacy_customer_code,service_region_cluster,crm_reference_number,marketing_batch_id,churn_flag
0,100000,56,Male,Pune,90,Family,Enterprise,1744.837483,Month-to-Month,1,45.911343,581.834169,78,0,2.235690,NaN,1,0,43.145340,2,12,9408,28.296724,2,1,0,43710.276247,1744.837483,1,682667,32,799034,8963,0
1,100001,69,Female,Hyderabad,73,Family,Enterprise,1191.787428,Month-to-Month,3,30.574466,1205.185406,72,1,0.856839,39.639547,1,3,28.007408,2,19,38319,67.890868,2,1,0,70444.095330,1191.787428,0,266664,34,160389,4153,0
2,100002,46,Female,Delhi,30,Consumer,Prepaid,1301.043042,2 Year,4,36.853273,297.478047,84,1,0.757148,76.607516,1,2,38.026476,3,15,29415,37.944444,2,1,0,67156.246579,1301.043042,0,579857,4,594893,9288,0
3,100003,32,Female,Chennai,108,Consumer,Postpaid,1657.169192,2 Year,1,32.850096,433.856754,69,0,2.418240,56.356062,3,1,21.726140,0,11,41955,NaN,1,1,1,40032.005394,1657.169192,1,434398,45,563518,5734,0
4,100004,60,Female,Delhi,112,Premium,Postpaid,1618.871274,2 Year,5,45.966472,555.242419,76,0,9.319477,NaN,1,2,10.196770,3,14,23221,97.267934,0,0,0,45895.698714,1618.871274,0,893274,19,373531,6535,0


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50500 entries, 0 to 50499
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 50500 non-null  int64  
 1   customer_age                50500 non-null  int64  
 2   gender                      50500 non-null  object 
 3   city                        50500 non-null  object 
 4   customer_tenure_months      50500 non-null  int64  
 5   customer_segment            50500 non-null  object 
 6   plan_type                   50500 non-null  object 
 7   monthly_bill_amount         50500 non-null  float64
 8   contract_type               50500 non-null  object 
 9   number_of_services          50500 non-null  int64  
 10  monthly_data_usage_gb       50500 non-null  float64
 11  monthly_voice_minutes       50500 non-null  float64
 12  monthly_sms_count           50500 non-null  int64  
 13  international_usage_flag    505

In [3]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,50500.0,NaN,NaN,NaN,124997.002455,14438.16276,100000.0,112484.75,124998.5,137504.25,149999.0
customer_age,50500.0,NaN,NaN,NaN,48.54398,17.879237,18.0,33.0,48.0,64.0,79.0
gender,50500,2,Male,25305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,50500,7,Bangalore,7328,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_tenure_months,50500.0,NaN,NaN,NaN,60.02002,34.419617,1.0,30.0,60.0,90.0,119.0
customer_segment,50500,4,Premium,12724,NaN,NaN,NaN,NaN,NaN,NaN,NaN
plan_type,50500,6,Postpaid,12887,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_bill_amount,50500.0,NaN,NaN,NaN,1207.075,491.387861,199.0,866.59985,1203.220589,1540.623502,3114.891086
contract_type,50500,3,Month-to-Month,27763,NaN,NaN,NaN,NaN,NaN,NaN,NaN
number_of_services,50500.0,NaN,NaN,NaN,2.99604,1.411713,1.0,2.0,3.0,4.0,5.0


### 1.1 Missing values

In [4]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})


,missing_count,missing_pct
average_download_speed,5139,10.18
campaign_response_rate,5046,9.99


`average_download_speed` and `campaign_response_rate` have ~10% missing values each. We keep these as `NaN` here (do **not** impute yet) — imputation is a *fitted* step and must wait until after the train-test split so the imputation statistic comes only from the training data.

### 1.2 Duplicate rows

In [6]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated customer_id values:", df['customer_id'].duplicated().sum())

df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)


Fully duplicated rows: 500
Duplicated customer_id values: 500


Shape after dropping duplicates: (50000, 34)


Duplicate removal is done **before** the split, because it operates on fixed row-level identity (an exact duplicate row) rather than any statistic learned from the data — there is no leakage risk here.

## 2. Standardize inconsistent categorical labels

In [5]:
for col in ["gender", "city", "customer_segment", "plan_type", "contract_type"]:
    print(col, "->", sorted(df[col].unique()))


gender -> ['Female', 'Male']
city -> ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune']
customer_segment -> ['Business', 'Consumer', 'Family', 'Premium']
plan_type -> ['Enterprise', 'POSTPAID', 'Postpaid', 'Prepaid', 'Unlimited', 'postpaid']
contract_type -> ['1 Year', '2 Year', 'Month-to-Month']


In [6]:
# plan_type has inconsistent casing: 'Postpaid', 'postpaid', 'POSTPAND'/'POSTPAID' all mean the same plan
df["plan_type"] = df["plan_type"].str.strip().str.title()
print(df["plan_type"].value_counts())


plan_type
Postpaid      14198
Unlimited     12268
Prepaid       12198
Enterprise    11836
Name: count, dtype: int64


This is a pure text-normalization fix (case/whitespace), not a statistical transformation, so it is safe to apply before splitting — it does not use any information that varies between train and test.

## 3. Handle sentinel / error codes

In [7]:
# monthly_data_usage_gb has a hard-coded sentinel value of 5000 GB which is physically
# implausible (real usage tops out around ~143 GB) and is almost certainly a data-capture
# error code rather than a genuine reading.
print((df["monthly_data_usage_gb"] == 5000).sum(), "rows flagged with sentinel value 5000")
print(df.loc[df["monthly_data_usage_gb"] != 5000, "monthly_data_usage_gb"].describe())

df["monthly_data_usage_gb"] = df["monthly_data_usage_gb"].replace(5000, np.nan)


101 rows flagged with sentinel value 5000
count    50399.000000
mean        39.959330
std         17.837168
min          1.282418
25%         26.920586
50%         37.354757
75%         50.178669
max        143.252950
Name: monthly_data_usage_gb, dtype: float64


Replacing a *known, fixed* sentinel code (5000) with `NaN` is a deterministic rule, not something fit on the data — so it's safe to apply pre-split. The actual **imputation** of the resulting missing values will be fit on the training set only, in the feature-engineering notebook.

## 4. Drop non-predictive / redundant columns

In [8]:
# monthly_bill_amount and monthly_revenue are perfectly correlated (identical) — keep one.
print("Correlation:", df['monthly_bill_amount'].corr(df['monthly_revenue']))
print("Identical values:", (df['monthly_bill_amount'] == df['monthly_revenue']).mean())

id_like_cols = ["customer_id", "legacy_customer_code", "crm_reference_number", "marketing_batch_id"]
redundant_cols = ["monthly_bill_amount"]  # duplicate of monthly_revenue

df = df.drop(columns=id_like_cols + redundant_cols)
print("Shape after dropping ID/redundant columns:", df.shape)


Correlation: 1.0
Identical values: 1.0
Shape after dropping ID/redundant columns: (50500, 29)


- `customer_id`, `legacy_customer_code`, `crm_reference_number`, `marketing_batch_id` are arbitrary identifiers with near-zero correlation to churn — they carry no predictive signal and risk overfitting/leaking row identity into the model.
- `monthly_bill_amount` is a byte-for-byte duplicate of `monthly_revenue`; keeping both artificially inflates multicollinearity (both showed infinite VIF). We keep `monthly_revenue` and drop the duplicate.

## 5. Train-Test Split (before any fitted preprocessing)

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["churn_flag"])
y = df["churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_df = X_train.copy()
train_df["churn_flag"] = y_train

test_df = X_test.copy()
test_df["churn_flag"] = y_test

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train churn rate:", y_train.mean().round(4))
print("Test churn rate:", y_test.mean().round(4))


Train shape: (40400, 29)
Test shape: (10100, 29)
Train churn rate: 0.2694
Test churn rate: 0.2693


## 6. Save split, structurally-cleaned data

In [10]:
train_df.to_csv(PROCESSED_DIR / "train_raw.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_raw.csv", index=False)

print("Saved:")
print(" -", PROCESSED_DIR / "train_raw.csv")
print(" -", PROCESSED_DIR / "test_raw.csv")


Saved:
 - ..\data\processed\train_raw.csv
 - ..\data\processed\test_raw.csv


**Note:** Full EDA (distributions, churn drivers, correlation heatmaps) should now be run on `train_raw.csv` only — see `02_eda.ipynb`. Never explore the test set visually before modeling; that is itself a mild form of leakage (it can subconsciously influence feature-engineering decisions).